# Créer un Vector Store
---

In [1]:
# Installer Faiss (version CPU)
!pip install faiss-cpu --quiet

# Installer le client Mistral
!pip install mistralai --quiet

In [1]:
import os
os.environ["MISTRAL_API_KEY"] = "StthE16bt5IReMHPxgjbwtQLGggjOGtq"

In [3]:
import mistralai.client
print(dir(mistralai.client))

['Any', 'AsyncHttpClient', 'BaseSDK', 'Callable', 'ClientOwner', 'Dict', 'Field', 'HttpClient', 'Logger', 'Mistral', 'OPENAPI_DOC_VERSION', 'Optional', 'OptionalNullable', 'RetryConfig', 'SDKConfiguration', 'SDKHooks', 'SERVERS', 'SERVER_EU', 'SPEAKEASY_GENERATOR_VERSION', 'TYPE_CHECKING', 'Tuple', 'UNSET', 'USER_AGENT', 'Union', 'VERSION', '__annotations__', '__builtins__', '__cached__', '__doc__', '__file__', '__gen_version__', '__loader__', '__name__', '__openapi_doc_version__', '__package__', '__path__', '__spec__', '__title__', '__user_agent__', '__version__', '_hooks', '_version', 'basesdk', 'cast', 'close_clients', 'dataclass', 'errors', 'get_default_logger', 'httpclient', 'httpx', 'importlib', 'models', 'models_', 'remove_suffix', 'sdk', 'sdkconfiguration', 'sys', 'types', 'utils', 'weakref']


In [ ]:
from mistralai.client import Mistral
import os

api_key = os.environ.get("MISTRAL_API_KEY")
client = MistralClient(api_key=api_key)

print("Ça marche !")

# Fonction pour générer l'embedding d'un texte
def embed_text(text):
    # Utilisation de la méthode 'create' de l'objet 'embeddings'
    response = client.embeddings.create(
        model="mistral-embed",
        inputs=[text]
    )
    return response.data[0].embedding

# Exemple de documents à indexer
documents = [
    "La mairie est ouverte du lundi au vendredi de 9h à 17h.",
    "Le règlement municipal interdit le stationnement devant les écoles.",
    "Les horaires des déchetteries varient selon les saisons.",
    "Les demandes de passeport se font uniquement sur rendez-vous.",
    "Les aides sociales sont disponibles sous conditions de revenus."
]

# Génération des embeddings pour chaque document
embeddings = np.array([embed_text(doc) for doc in documents])

ImportError: cannot import name 'MistralClient' from 'mistralai.client' (/home/camille_sc/Bureau/Formation/Projet_11/.venv/lib/python3.13/site-packages/mistralai/client/__init__.py)

In [ ]:
import faiss

# Dimension des vecteurs (embedding size)
dimension = embeddings.shape[1]

# Création d'un index Faiss L2 (distance euclidienne)
index = faiss.IndexFlatL2(dimension)

# Ajout des vecteurs à l'index
index.add(embeddings)

# Sauvegarde éventuelle
faiss.write_index(index, "faiss_index.idx")

In [ ]:
# Requête utilisateur
query = "Quels sont les horaires d'ouverture de la mairie ?"

# Génération de l'embedding de la requête
query_embedding = embed_text(query)

# Recherche des 3 documents les plus proches
k = 3
distances, indices = index.search(np.array([query_embedding]), k)

# Affichage des résultats
print("Résultats les plus pertinents :\n")
for i, idx in enumerate(indices[0]):
    print(f"📝 Document {idx} (distance: {distances[0][i]:.4f}) :\n{documents[idx]}\n")